# B4 · LLM en pratique : Hugging Face, prompting, RAG et LoRA — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b4_llm/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B4.Q1 à B4.Q12, B4.R1 à B4.R3, B4.1 à B4.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B4.12 à B4.16 | usage et prompting : premier contact avec transformers, générations à trois températures, chat template et streaming, extraction en JSON validé, sensibilité au prompt (*prochaine session de génération*) | | | |
| B | B4.17 à B4.24 | recherche et RAG : chunk_text et BM25, cosine_top_k, RRF et métriques de ta librairie, BM25 contre recherche dense, RAG avec citations, quatre défauts, évaluation, injection (*prochaine session de génération*) | | | |
| C | B4.25 à B4.28 | fine-tuning : LoRA à la main contre peft, style de Holmes, rang et modules ciblés, classification de textes (*prochaine session de génération*) | | | |
| D | B4.29 et B4.30 | livraison : pipeline reproductible et défi RAG (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Estimer la mémoire d'un LLM (poids selon le dtype, KV cache) et appliquer à la main son chat template.
- Charger et faire générer de petits modèles avec transformers, écrire et évaluer des prompts sur plusieurs formulations et graines.
- Implémenter le découpage, BM25, la recherche dense, la fusion RRF, recall@k et MRR, puis construire, évaluer et diagnostiquer un RAG avec citations.
- Compter, écrire à la main et appliquer avec peft une adaptation LoRA, et comparer fine-tuning, prompting et classifieur classique.

**Rappel express.** Mémoire des poids $\approx N \times$ octets par paramètre (4 en float32, 2 en bfloat16, $\approx 0{,}5$ en int4) ; KV cache $= 2 \times L \times T \times n_{kv} d_h \times$ octets par valeur. BM25 : $\sum_{t \in q} \operatorname{idf}(t)\,\frac{f_{t,d}(k_1 + 1)}{f_{t,d} + k_1(1 - b + b\,|d| / \overline{|d|})}$, $\operatorname{idf}(t) = \ln\big(1 + \frac{N - n_t + 0{,}5}{n_t + 0{,}5}\big)$. RRF : $\sum_r 1/(k + \operatorname{rang}_r(d))$, $k = 60$. recall@$k$ $= |\text{top-}k \cap \text{pertinents}| / |\text{pertinents}|$ ; MRR = moyenne de $1/\operatorname{rang}$ du premier pertinent. LoRA : $\mathbf{W}' = \mathbf{W} + \frac{\alpha}{r}\mathbf{B}\mathbf{A}$, $\mathbf{B}_0 = 0$, $r(d + k)$ paramètres au lieu de $dk$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B4.1 à B4.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un classement : une liste d'identifiants, du meilleur au moins bon (`[5, 6, 4]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B4.6, l'estimation 🧮 B4.7, la réflexion (🗣️ B4.8, ⚖️ B4.9 et B4.10, 📄 B4.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math
from fractions import Fraction

import numpy as np

MB, GB = 10 ** 6, 10 ** 9                                           # the units of the chapter (decimal)

# ✏️ B4.1: SmolLM2-135M-Instruct (config.json, revision 12fd25f)
N_B41, LAYERS_B41, D_B41, V_B41, KV_HEADS_B41, DH_B41, CTX_B41 = 134_515_008, 30, 576, 49_152, 3, 64, 8_192
EMB_B41 = V_B41 * D_B41                                             # the token table (tied with the output layer)
EMB_SHARE_B41 = 100 * EMB_B41 / N_B41
FP32_B41, BF16_B41, INT4_B41 = 4 * N_B41 / MB, 2 * N_B41 / MB, 0.5 * N_B41 / MB
KV_TOKEN_B41 = 2 * LAYERS_B41 * KV_HEADS_B41 * DH_B41 * 2           # keys and values, every layer, bf16
KV_FULL_B41 = KV_TOKEN_B41 * CTX_B41 / MB
CONVERSATIONS_B41 = int((2 * GB - 2 * N_B41) // (KV_TOKEN_B41 * CTX_B41))
QWEN_FOOTPRINT, QWEN_SECONDS, QWEN_TOKENS = 988_065_792, 7.6, 64     # fiche §204.2, Qwen2.5-0.5B-Instruct in bf16
SPEED_B41 = QWEN_TOKENS / QWEN_SECONDS * QWEN_FOOTPRINT / (2 * N_B41)

# ✏️ B4.2: five queries, five retrieved ids each
RETRIEVED_B42 = [[3, 1, 4, 0, 2], [5, 9, 2, 6, 8], [0, 7, 1, 2, 3], [6, 4, 8, 9, 1], [2, 0, 9, 3, 7]]
RELEVANT_B42 = [{1}, {2, 8}, {0, 5}, {7}, {3, 4, 9}]


def recall_b42(k, retrieved=RETRIEVED_B42, relevant=RELEVANT_B42, denominator="relevant"):
    """Mean of |top-k ∩ relevant| / |relevant| (denominator="k": the precision@k mistake)."""
    ratios = [len(set(r[:k]) & rel) / (len(rel) if denominator == "relevant" else k)
              for r, rel in zip(retrieved, relevant)]
    return sum(ratios) / len(ratios)


def first_rank_b42(r, rel):
    return next((i + 1 for i, doc in enumerate(r) if doc in rel), None)


MRR_B42 = sum(1 / rank if (rank := first_rank_b42(r, rel)) else 0
              for r, rel in zip(RETRIEVED_B42, RELEVANT_B42)) / 5
HIT3_B42 = sum(any(doc in rel for doc in r[:3]) for r, rel in zip(RETRIEVED_B42, RELEVANT_B42)) / 5
FULL5_B42 = sum(set(r[:5]) >= rel for r, rel in zip(RETRIEVED_B42, RELEVANT_B42))

# ✏️ B4.3: LoRA on Qwen2.5-0.5B-Instruct (24 layers, d_model = 896, k and v 896 -> 128, MLP 896 -> 4 864)
N_QWEN, LAYERS_QWEN, D_QWEN, KV_QWEN, MLP_QWEN, R_B43 = 494_032_768, 24, 896, 128, 4_864, 8


def lora_params(d_out, d_in, r=R_B43):
    return r * (d_out + d_in)


Q_B43, K_B43 = lora_params(D_QWEN, D_QWEN), lora_params(KV_QWEN, D_QWEN)
Q_SHARE_B43, K_SHARE_B43 = 100 * Q_B43 / D_QWEN ** 2, 100 * K_B43 / (KV_QWEN * D_QWEN)
LAYER_B43 = 2 * Q_B43 + 2 * K_B43                                   # q, k, v, o
MODEL_B43 = LAYERS_QWEN * LAYER_B43
TRAINABLE_PCT_B43 = 100 * MODEL_B43 / (N_QWEN + MODEL_B43)
ALL_LINEAR_B43 = LAYERS_QWEN * (LAYER_B43 + 3 * lora_params(MLP_QWEN, D_QWEN))
ADAPTER_MB_B43 = 4 * MODEL_B43 / MB
NO_SAVING_RANK_B43 = math.ceil(D_QWEN ** 2 / (2 * D_QWEN))           # r (d + k) >= d k

# ✏️ B4.4: the chat template of SmolLM2-135M-Instruct (revision 12fd25f), and the tokens of each piece
SMOL_TEMPLATE = ("{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}"
                 "{{ '<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face"
                 "<|im_end|>\n' }}{% endif %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + "
                 "'<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}"
                 "{% endif %}")
DEFAULT_SYSTEM = "You are a helpful AI assistant named SmolLM, trained by Hugging Face"
S_B44 = {"role": "system", "content": "Answer in one short sentence."}
U_B44 = {"role": "user", "content": "Where does Holmes live?"}
A_B44 = {"role": "assistant", "content": "In London, at 221B Baker Street."}
U2_B44 = {"role": "user", "content": "Who is his friend?"}
CONTENT_TOKENS = {DEFAULT_SYSTEM: 16, S_B44["content"]: 6, U_B44["content"]: 5, A_B44["content"]: 12,
                  U2_B44["content"]: 5}
ROLE_TOKENS = {"system": 1, "user": 1, "assistant": 2}


def by_hand_b44(messages, add_generation_prompt):
    """The chat template applied by hand: the string, its special tokens and its token count."""
    if messages[0]["role"] != "system":
        messages = [{"role": "system", "content": DEFAULT_SYSTEM}] + messages
    text = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    specials = 2 * len(messages)
    tokens = sum(1 + ROLE_TOKENS[m["role"]] + 1 + CONTENT_TOKENS[m["content"]] + 1 + 1 for m in messages)
    if add_generation_prompt:
        text, specials, tokens = text + "<|im_start|>assistant\n", specials + 1, tokens + 1 + 2 + 1
    return text, specials, tokens


U_GEN_B44 = by_hand_b44([U_B44], True)
SU_GEN_B44, SU_NOGEN_B44 = by_hand_b44([S_B44, U_B44], True), by_hand_b44([S_B44, U_B44], False)
TURN2_B44 = by_hand_b44([S_B44, U_B44, A_B44, U2_B44], True)
DOUBLE_SYSTEM_B44 = by_hand_b44([{"role": "system", "content": DEFAULT_SYSTEM}, S_B44, U_B44], True)

# ✏️ B4.5: BM25 by hand (k1 = 1.5, b = 0.75), three documents, query "nemo nautilus"
DOCS_B45 = [["nemo", "commande", "le", "nautilus"], ["le", "nautilus", "plonge"],
            ["le", "nautilus", "croise", "un", "nautilus", "au", "grand", "large"]]
QUERY_B45 = ["nemo", "nautilus"]


def bm25_b45(query=QUERY_B45, docs=DOCS_B45, k1=1.5, b=0.75, idf_form="plus_one"):
    """BM25 scores of the docstring; idf_form="original" drops the "1 +" of the idf."""
    n, avgdl = len(docs), sum(map(len, docs)) / len(docs)
    scores = np.zeros(n)
    for t in query:
        n_t = sum(t in doc for doc in docs)
        ratio = (n - n_t + 0.5) / (n_t + 0.5)
        idf = math.log(1 + ratio) if idf_form == "plus_one" else math.log(ratio)
        for i, doc in enumerate(docs):
            f = doc.count(t)
            if f:
                scores[i] += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(doc) / avgdl))
    return scores


AVGDL_B45 = sum(map(len, DOCS_B45)) / 3
IDF_NAUTILUS_B45 = math.log(1 + 0.5 / 3.5)
SCORES_B45 = bm25_b45()
RANKING_B45 = [int(i) for i in np.argsort(-SCORES_B45, kind="stable")]
SCORES_B0_B45 = bm25_b45(b=0.0)
SCORES_K0_B45 = bm25_b45(k1=0.0)
SCORES_ORIGINAL_B45 = bm25_b45(idf_form="original")

# quizzes and recalls with a computation
Q3A, Q3B = 2 * 3e9 / GB, 0.5 * 3e9 / GB
Q8C = math.sqrt(0.8 * 0.2 / 100)
Q9E = 2 / (60 + 3)
Q11C = 4 * (512 + 512)
R1C = 2 * 24 * 2 * 64 * 2
A_R3, B_R3 = np.array([3.0, 4.0]), np.array([6.0, 8.0])
COS_R3 = float(A_R3 @ B_R3 / (np.linalg.norm(A_R3) * np.linalg.norm(B_R3)))
DIST_R3 = float(np.linalg.norm(A_R3 - B_R3))
SQ_R3 = 2 - 2 * 0.82
MULT_R3 = 50_000 * 768

# The same answers from the libraries (no download): mylearn (reference), scikit-learn, transformers and peft
import torch
from peft import LoraConfig, get_peft_model
from torch import nn
from transformers import AutoModelForCausalLM, LlamaConfig, Qwen2Config
from transformers.utils.chat_template_utils import render_jinja_template

R = mylearn.retrieval
# ✏️ B4.2: recall_at_k and mean_reciprocal_rank of the reference
for k, expected in ((1, recall_b42(1)), (3, recall_b42(3)), (5, recall_b42(5))):
    assert math.isclose(R.recall_at_k(RETRIEVED_B42, RELEVANT_B42, k=k), expected)
assert math.isclose(R.mean_reciprocal_rank(RETRIEVED_B42, RELEVANT_B42), MRR_B42)
assert R.mean_reciprocal_rank(RETRIEVED_B42, RELEVANT_B42) >= R.recall_at_k(RETRIEVED_B42, RELEVANT_B42, k=1)

# ✏️ B4.5: BM25 of the reference, then the same corpus with b = 0, k1 = 0 and the original idf
bm25 = R.BM25(k1=1.5, b=0.75).fit(DOCS_B45)
assert bm25.avgdl_ == AVGDL_B45 and math.isclose(bm25.idf_["nautilus"], IDF_NAUTILUS_B45)
assert np.allclose(bm25.get_scores(QUERY_B45), SCORES_B45)
assert np.allclose(R.BM25(k1=1.5, b=0.0).fit(DOCS_B45).get_scores(QUERY_B45), SCORES_B0_B45)
assert np.allclose(R.BM25(k1=0.0, b=0.75).fit(DOCS_B45).get_scores(QUERY_B45), SCORES_K0_B45)
assert SCORES_ORIGINAL_B45[0] < 0                                   # below a document with no query word (score 0)
assert RANKING_B45 == [int(i) for i in np.argsort(-bm25.get_scores(QUERY_B45), kind="stable")]

# quiz Q9 e: a document ranked 3rd in two rankings (reciprocal_rank_fusion only returns the ids: the score by hand)
assert R.reciprocal_rank_fusion([[1, 2, 7], [3, 4, 7]])[0] == 7 and math.isclose(float(Fraction(2, 63)), Q9E)

# rappel R3: cosine and Euclidean distance, scikit-learn and the B2 reference
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
assert math.isclose(cosine_similarity([A_R3], [B_R3])[0, 0], COS_R3)
assert math.isclose(mylearn.embeddings.cosine_similarity_matrix([A_R3], [B_R3])[0, 0], COS_R3)
assert math.isclose(euclidean_distances([A_R3], [B_R3])[0, 0], DIST_R3)
u = np.array([1.0, 0.0]); v = np.array([0.82, math.sqrt(1 - 0.82 ** 2)])
assert math.isclose(float(np.sum((u - v) ** 2)), SQ_R3)

# ✏️ B4.1: SmolLM2-135M built from its configuration on the meta device (shapes only, no download)
smol_config = LlamaConfig(hidden_size=576, intermediate_size=1536, num_hidden_layers=30, num_attention_heads=9,
                          num_key_value_heads=3, vocab_size=49152, max_position_embeddings=8192,
                          tie_word_embeddings=True)
with torch.device("meta"):
    smol = AutoModelForCausalLM.from_config(smol_config)
assert smol.num_parameters() == N_B41 and smol.model.embed_tokens.weight.numel() == EMB_B41
assert smol.lm_head.weight is smol.model.embed_tokens.weight
head_dim = smol_config.hidden_size // smol_config.num_attention_heads
assert 2 * smol_config.num_hidden_layers * smol_config.num_key_value_heads * head_dim * 2 == KV_TOKEN_B41

# ✏️ B4.3 and rappel R1: Qwen2.5-0.5B built from its configuration, then wrapped by peft (r = 8)
qwen_config = Qwen2Config(hidden_size=896, intermediate_size=4864, num_hidden_layers=24, num_attention_heads=14,
                          num_key_value_heads=2, vocab_size=151936, max_position_embeddings=32768,
                          tie_word_embeddings=True)


def peft_counts(target_modules, r=8):
    with torch.device("meta"):
        base = AutoModelForCausalLM.from_config(qwen_config)
    return get_peft_model(base, LoraConfig(r=r, lora_alpha=16, target_modules=target_modules, lora_dropout=0.0,
                                           task_type="CAUSAL_LM")).get_nb_trainable_parameters()


with torch.device("meta"):
    qwen = AutoModelForCausalLM.from_config(qwen_config)
assert qwen.num_parameters() == N_QWEN
attention = qwen.model.layers[0].self_attn
assert attention.k_proj.weight.shape == (KV_QWEN, D_QWEN) and attention.q_proj.weight.shape == (D_QWEN, D_QWEN)
trainable, total = peft_counts(["q_proj", "k_proj", "v_proj", "o_proj"])
assert trainable == MODEL_B43 and total == N_QWEN + MODEL_B43
assert math.isclose(100 * trainable / total, TRAINABLE_PCT_B43)
assert peft_counts("all-linear")[0] == ALL_LINEAR_B43
assert peft_counts(["q_proj", "k_proj", "v_proj", "o_proj"], r=16)[0] == 2 * MODEL_B43
layer = nn.Sequential(nn.Linear(D_QWEN, D_QWEN))                     # one 896 x 896 layer, as in a)
assert get_peft_model(layer, LoraConfig(r=8, target_modules=["0"])).get_nb_trainable_parameters()[0] == Q_B43
assert 2 * qwen_config.num_hidden_layers * qwen_config.num_key_value_heads * 64 * 2 == R1C

# ✏️ B4.4: the template rendered by transformers' own Jinja renderer, as apply_chat_template does
def render_b44(messages, add_generation_prompt):
    return render_jinja_template(conversations=[messages], chat_template=SMOL_TEMPLATE,
                                 add_generation_prompt=add_generation_prompt)[0][0]


for messages, gen, by_hand in (([U_B44], True, U_GEN_B44), ([S_B44, U_B44], True, SU_GEN_B44),
                               ([S_B44, U_B44], False, SU_NOGEN_B44), ([S_B44, U_B44, A_B44, U2_B44], True, TURN2_B44)):
    rendered = render_b44(messages, gen)
    assert rendered == by_hand[0], (rendered, by_hand[0])
    assert rendered.count("<|im_start|>") + rendered.count("<|im_end|>") == by_hand[1]
try:                                                                 # optional: the real tokenizer, if already cached
    from transformers import AutoTokenizer
    smol_tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct",
                                             revision="12fd25f77366fa6b3b4b768ec3050bf629380bac", local_files_only=True)
except OSError:
    print("ℹ️ SmolLM2's tokenizer is not in the local cache: token counts checked when the chapter was generated")
else:
    assert smol_tok.chat_template == SMOL_TEMPLATE
    for messages, gen, by_hand in (([U_B44], True, U_GEN_B44), ([S_B44, U_B44], True, SU_GEN_B44),
                                   ([S_B44, U_B44], False, SU_NOGEN_B44),
                                   ([S_B44, U_B44, A_B44, U2_B44], True, TURN2_B44)):
        ids = smol_tok.apply_chat_template(messages, add_generation_prompt=gen, return_dict=True)["input_ids"]
        assert len(ids) == by_hand[2] and len(smol_tok(by_hand[0])["input_ids"]) == by_hand[2]
    print("✅ token counts of ✏️ B4.4 checked with the tokenizer of SmolLM2-135M-Instruct (local cache)")
print("✅ the paper answers agree with mylearn (reference), scikit-learn, transformers and peft")

/root/venv313/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ token counts of ✏️ B4.4 checked with the tokenizer of SmolLM2-135M-Instruct (local cache)
✅ the paper answers agree with mylearn (reference), scikit-learn, transformers and peft


**B4.Q1 — Modèle de base, modèle instruct, modèle de chat : lequel pour quoi ?**

In [3]:
wb.record("B4.Q1a", "B", mistakes={"relis « Trois étapes, trois sortes de modèles » : ce que l'on veut ici est-il un dialogue ?": "A", "relis l'étape 1 de « Trois étapes » : quel modèle continue n'importe quel texte ?": "C"})
wb.record("B4.Q1b", True, mistakes={"compare les tokens du pré-entraînement et les paires du SFT (§204.1)": False})
wb.record("B4.Q1c", "C", mistakes={"relis le paragraphe sur les noms du Hub : Qwen3 fait l'inverse des autres": "A", "relis le paragraphe sur les noms du Hub : que signifie « -it » chez Google ?": "B"})
wb.record("B4.Q1d", "C", mistakes={"relis l'étape 1 de « Trois étapes » : que produit le pré-entraînement ?": "A", "un gabarit met en forme une conversation : apprend-il au modèle à suivre une consigne ?": "B"})
wb.record("B4.Q1e", False, mistakes={"relis l'étape 1 de « Trois étapes » : qu'est-ce qui peut suivre une question dans un texte ?": True})

WB_ANSWER {"hash": "587da99447b97618dc09e245f5c97f8f08a773df86aa582d43b0809aa0743859", "id": "B4.Q1a", "kind": "str", "mistakes": {"5a6f4b408d6f08b5a0e20cc87c0abe4031a777999be5d855d4a37eab30c8a4d3": "relis l'étape 1 de « Trois étapes » : quel modèle continue n'importe quel texte ?", "e27067e146062c0eb17baed35605f67fec3a50c698999babe860d71f87631800": "relis « Trois étapes, trois sortes de modèles » : ce que l'on veut ici est-il un dialogue ?"}}
📝 Ex B4.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "0bd64138fcee23b52a3c181993b92b53a8e6be0fdebf3f84982748250e482485", "id": "B4.Q1b", "kind": "bool", "mistakes": {"8331be05f668c156fae54beaefdd86f68eba974f07c0d52382ce01cb8910a7d4": "compare les tokens du pré-entraînement et les paires du SFT (§204.1)"}}
📝 Ex B4.Q1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "a364087b4aa9c846bb29559d95ad2984751c5752de7fe8f22766df98fef58239", "id": "B4.Q1c", "kind": "str", "mistakes": {"21bc18e3a212e7a987baf3db89a6bfa2d6d86a0e098e215aa5cdd354096472c0

**B4.Q2 — « Open weights » n'est pas « open source » : lire la licence d'un modèle**

In [4]:
wb.record("B4.Q2a", True, mistakes={"relis la première puce de « Licences » (§204.1)": False})
wb.record("B4.Q2b", "B", mistakes={"la licence d'origine impose-t-elle une autre licence aux dérivés ? Relis « Licences »": "A", "relis ce que la licence de Llama 3.2 demande pour un modèle dérivé redistribué": "C"})
wb.record("B4.Q2c", False, mistakes={"relis le paragraphe sur l'Open Source AI Definition : que demande-t-elle en plus des poids ?": True})
wb.record("B4.Q2d", "C", mistakes={"relis la troisième puce de « Licences » : que faut-il pour un accès contrôlé ?": "A", "un miroir non autorisé respecte-t-il les conditions de l'éditeur ?": "B"})
wb.record("B4.Q2e", "A", mistakes={"transformers exécute ces modèles : le code d'inférence est-il caché ?": "B", "from_pretrained lit config.json : ce fichier manque-t-il ?": "C"})

WB_ANSWER {"hash": "63f9c5e258e9d96fba2965fffc74929a37377e030cf72cdeb7bfc217bf381bae", "id": "B4.Q2a", "kind": "bool", "mistakes": {"327a85745b0cd2e16a611ff822d541c91bf5d1651cc0ffb7f5e319628ad64cb9": "relis la première puce de « Licences » (§204.1)"}}
📝 Ex B4.Q2a : réponse enregistrée (bool).
WB_ANSWER {"hash": "eec6fe80b7b7895cb2b8ceef2b4fdf9f4f56ad98bde5f2819aea9e68c757febb", "id": "B4.Q2b", "kind": "str", "mistakes": {"ad8078928d0e4c4314e806cb00d198eb0419b3c6546996d05997d20fa53b767d": "relis ce que la licence de Llama 3.2 demande pour un modèle dérivé redistribué", "ce68c8d3026aade93d7996c56587fd50bc01eadde2b8029b855a56fff87459fd": "la licence d'origine impose-t-elle une autre licence aux dérivés ? Relis « Licences »"}}
📝 Ex B4.Q2b : réponse enregistrée (str).
WB_ANSWER {"hash": "7beaf06da6645993dce709809ca5c24cb644a83755e8d360f31a3d1773f8e3e1", "id": "B4.Q2c", "kind": "bool", "mistakes": {"f5d5f326298ab9f4b8def03132290feed34c1498099c6ea312889876465191d4": "relis le paragraphe sur l

**B4.Q3 — dtype, device et mémoire : ce qui décide si un modèle tient sur ton ordinateur**

In [5]:
wb.record("B4.Q3a", Q3A, decimals=1, mistakes={"combien d'octets par paramètre en bfloat16 ? Relis « La mémoire des poids »": 12.0,
          "relis la question : bfloat16, pas int4": 1.5, "1 Go = 10⁹ octets : vérifie la conversion": 6000.0})
wb.record("B4.Q3b", Q3B, decimals=1, mistakes={"relis la question : int4, à 0,5 octet par paramètre": 6.0,
          "combien d'octets par paramètre en int4 ? Relis « La mémoire des poids »": 3.0,
          "c'est le nombre de bits : on demande des Go": 12.0})
wb.record("B4.Q3c", True, mistakes={"relis l'encadré 🧮 sur float32 et bfloat16 : combien de bits d'exposant ?": False})
wb.record("B4.Q3d", "C", mistakes={"relis la formule du KV cache (§204.2) : V y apparaît-il ?": "A", "relis la formule du KV cache : quelle quantité grandit pendant la conversation ?": "B"})
wb.record("B4.Q3e", False, mistakes={"relis « Vitesse sur CPU » : compare les deux mesures de la fiche": True})

WB_ANSWER {"decimals": 1, "hash": "a34764beb49e4c7807d217f74d880fac407ef8bf61e06eaf406e8352df375476", "hash_coarse": "739b63803708fd37c1165e6ce21cf397489fb5eee927b30a0cb9d5751750c52d", "id": "B4.Q3a", "kind": "float", "magnitude": 0, "mistakes": {"86bbe2bd375a466ed2cb3a57cfbfa00c26e8706c26b5a6918fd6ead43f537640": "1 Go = 10⁹ octets : vérifie la conversion", "8ed8aa4f47099d582fe861081019a1f68864659b02cbbccbae2771a95ff34fa4": "relis la question : bfloat16, pas int4", "ad877b4b7858c813a6b95db51ef8c2fc138640d9bfad693b87ae8cef2fb40767": "combien d'octets par paramètre en bfloat16 ? Relis « La mémoire des poids »"}, "sign": 1}
📝 Ex B4.Q3a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "50788f375d8ae673bd464f95e521514bdd79c4eb1c9f4e9a96510a1975b689ea", "hash_coarse": "a3c105d04bc36354a54a682c165c5959e06bdf953c534fe77fd78bff82289baa", "id": "B4.Q3b", "kind": "float", "magnitude": 0, "mistakes": {"206bb6f35e23aa38df871faacea73a664ad43713e2d4a49ba25e5eef5d83cc59"

**B4.Q4 — Chat template : pourquoi ne pas écrire soi-même « User: … Assistant: … »**

In [6]:
wb.record("B4.Q4a", "A", mistakes={"relis « Rôles et chat template » : où le gabarit est-il rangé ?": "B", "generation_config.json règle la génération : relis « Rôles et chat template »": "C"})
wb.record("B4.Q4b", True, mistakes={"relis la première des trois choses à retenir (§204.3)": False})
wb.record("B4.Q4c", "C", mistakes={"le tokenizer réécrit-il le texte que tu lui donnes ? Relis le paragraphe qui suit les trois choses à retenir": "A", "relis le paragraphe qui suit les trois choses à retenir : le modèle a-t-il vu ce format ?": "B"})
wb.record("B4.Q4d", False, mistakes={"relis la fin du paragraphe sur le format écrit à la main (§204.3)": True})
wb.record("B4.Q4e", "B", mistakes={"skip_special_tokens sert au décodage : relis la troisième chose à retenir": "A", "add_generation_prompt agit sur la fin de la chaîne : relis la troisième chose à retenir": "C"})

WB_ANSWER {"hash": "a0f2240b84e8ee4ccbc962f8d9d48fe9f48671dbb1a4df0fee4f860068179fe7", "id": "B4.Q4a", "kind": "str", "mistakes": {"a9942ee1e7ab388fc91ccd31538a24905d6769cc259592177a2b6973de60ae6c": "relis « Rôles et chat template » : où le gabarit est-il rangé ?", "ef186bf266e7cfdcb734fd3cbe8fc3aab46104b5dd4a6b50af7129faa39aedc2": "generation_config.json règle la génération : relis « Rôles et chat template »"}}
📝 Ex B4.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "fb6584d54e9db743f561cac8a7f2afef0dda177c57814c6e7c6379465974cac7", "id": "B4.Q4b", "kind": "bool", "mistakes": {"233143e52219ff92e4cea7b478147edeb92d459e72ba6b9ea0a7234b21a9061c": "relis la première des trois choses à retenir (§204.3)"}}
📝 Ex B4.Q4b : réponse enregistrée (bool).
WB_ANSWER {"hash": "fc453497c3f06ad26c62d1667965376b538de75331c4f7889e713d2f62ad505e", "id": "B4.Q4c", "kind": "str", "mistakes": {"4559aa06a59fd943e5f6760bb0caff03a5ca9ba99ec1251b12800c520cc8256c": "relis le paragraphe qui suit les trois chos

**B4.Q5 — Glouton, température, top-p : lequel pour extraire une date, lequel pour écrire un poème ?**

In [7]:
wb.record("B4.Q5a", "C", mistakes={"une température haute rend-elle la sortie plus ou moins variable ?": "A", "une graine différente à chaque appel donne-t-elle la même réponse ?": "B"})
wb.record("B4.Q5b", "A", mistakes={"le décodage glouton donne-t-il dix réponses différentes ?": "B", "relis la troisième puce de « generate() et ses réglages » : une température de 0 se passe-t-elle ?": "C"})
wb.record("B4.Q5c", False, mistakes={"relis le piège classique sur generation_config.json (§204.3)": True})
wb.record("B4.Q5d", True, mistakes={"relis la température au ch. 22 : diviser les logits par 0,5 écarte-t-il ou rapproche-t-il les probabilités ?": False})
wb.record("B4.Q5e", "B", mistakes={"top-p compte des probabilités, pas une part du vocabulaire (B3)": "A", "top-p cumule des probabilités : relis B3, top-k et top-p": "C"})

WB_ANSWER {"hash": "24257ec5e713d1b8da3410b23aab149d65fd7bc8560e8511b2af46f2d9711353", "id": "B4.Q5a", "kind": "str", "mistakes": {"37a82be0d7af5cf31fa446d587459d31e4b773678f3f08e3cc4e3b36e10f6e80": "une graine différente à chaque appel donne-t-elle la même réponse ?", "a950d331406540c71bfa6e4ad82196757152a5c675c9d8f8e155f0d64029dedc": "une température haute rend-elle la sortie plus ou moins variable ?"}}
📝 Ex B4.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "189c80e0b2b917dd7177751a2014dbf9c7cbe9a42c5bc1d48e9854903d5a6e6c", "id": "B4.Q5b", "kind": "str", "mistakes": {"1158cb6a0a7ee2d01d85e4676f2a7eb4e86432a64f5cfdf73d8da8b8d5b8c397": "relis la troisième puce de « generate() et ses réglages » : une température de 0 se passe-t-elle ?", "e0e4a6cd7aac5ed2eac4f924bddd8e77de3b09b2bb854387a7f302a3bf498280": "le décodage glouton donne-t-il dix réponses différentes ?"}}
📝 Ex B4.Q5b : réponse enregistrée (str).
WB_ANSWER {"hash": "0af3665a00d792d76c5c776851ad83ec9e9b1342aada3f709aa24080d4

**B4.Q6 — Zéro-shot, few-shot, chain-of-thought : vrai ou faux**

In [8]:
wb.record("B4.Q6a", False, mistakes={"relis la définition de l'apprentissage en contexte (§204.4)": True})
wb.record("B4.Q6b", True, mistakes={"relis « Un prompt est fragile » : que montrent Lu et coll. ?": False})
wb.record("B4.Q6c", True, mistakes={"relis la fin du paragraphe « Zéro-shot et few-shot »": False})
wb.record("B4.Q6d", False, mistakes={"relis la première réserve sur le chain-of-thought (§204.4)": True})
wb.record("B4.Q6e", False, mistakes={"relis la seconde réserve sur le chain-of-thought (§204.4)": True})

WB_ANSWER {"hash": "5f03b717a6179fe92db953997a30fb129021bb1d8d5f87c5bd5774f5629ae56c", "id": "B4.Q6a", "kind": "bool", "mistakes": {"29f18792e5ceac9005df18eadd3881f06568fd3dfdff58c9062f807324070717": "relis la définition de l'apprentissage en contexte (§204.4)"}}
📝 Ex B4.Q6a : réponse enregistrée (bool).
WB_ANSWER {"hash": "8a4d94a30fb55235e099ac90b51e68061819225e6f7a56f8fc1f5f289629458b", "id": "B4.Q6b", "kind": "bool", "mistakes": {"d9f54ddca2e2c8eb51378dee1c74f3c649d0800987208404527bbca0ee5c55e5": "relis « Un prompt est fragile » : que montrent Lu et coll. ?"}}
📝 Ex B4.Q6b : réponse enregistrée (bool).
WB_ANSWER {"hash": "132c580c0a326998f76cf268080c225776c6a636036087744670b4aedf43b865", "id": "B4.Q6c", "kind": "bool", "mistakes": {"80bc066bf4fcf60df3dad040c30093e2bdfd4479e28684e10f5f31ceac2a2bf9": "relis la fin du paragraphe « Zéro-shot et few-shot »"}}
📝 Ex B4.Q6c : réponse enregistrée (bool).
WB_ANSWER {"hash": "1255c3ba0576bf9a60ca3322cd980b3fded89a7f6aa485e62355fd0eebd86873", "

**B4.Q7 — Pourquoi un LLM invente : hallucination et token le plus probable**

In [9]:
wb.record("B4.Q7a", "A", mistakes={"relis « Ce que fait vraiment un LLM » : consulte-t-il une base de faits ?": "B", "relis « Ce que fait vraiment un LLM » : comment une réponse entière se construit-elle ?": "C"})
wb.record("B4.Q7b", False, mistakes={"relis la dernière parade de « Les hallucinations » (§204.10)": True})
wb.record("B4.Q7c", "C", mistakes={"une année plausible pour un fait qui n'existe pas : est-ce juste ?": "A", "Holmes a-t-il pu recevoir un prix Nobel ? Relis les causes d'hallucination": "B"})
wb.record("B4.Q7d", True, mistakes={"relis les parades de « Les hallucinations » : un RAG garantit-il une réponse vraie ?": False})
wb.record("B4.Q7e", False, mistakes={"relis le premier paragraphe de « Les hallucinations » : qu'a récompensé l'entraînement ?": True})

WB_ANSWER {"hash": "77ee90a39dda63169bd938b70daeea397ae0469bb0372583fb6092231ea5e951", "id": "B4.Q7a", "kind": "str", "mistakes": {"00d05b64bb30872339b565165667f849fb7100c0dface629809154e79524eb4a": "relis « Ce que fait vraiment un LLM » : consulte-t-il une base de faits ?", "216fdc6b74e0cd7d2805fbc67b7474f32d9e600d2a1f0391a9a03038af0a61d2": "relis « Ce que fait vraiment un LLM » : comment une réponse entière se construit-elle ?"}}
📝 Ex B4.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "efa8c182aff4adcd338df641810b934215933c7bc3bd7a517217562d6a02c9d0", "id": "B4.Q7b", "kind": "bool", "mistakes": {"8e5192cb9de584fe82a17de66be56336b9d7a47b24d7981000e4242d46c9c3cf": "relis la dernière parade de « Les hallucinations » (§204.10)"}}
📝 Ex B4.Q7b : réponse enregistrée (bool).
WB_ANSWER {"hash": "ea3ffb08189f9b8069fce11a7ba7d36d606dd9c2b660e3c634d6feb7c1bceceb", "id": "B4.Q7c", "kind": "str", "mistakes": {"067f8148bdb9869bfaf9803a28724415d01aa3e0dbd1d5e7589433411306f211": "une année plausi

**B4.Q8 — Évaluer un LLM : exact match, juge LLM, contamination des benchmarks**

In [10]:
wb.record("B4.Q8a", 1, mistakes={"relis la normalisation de l'exact match : que deviennent la casse et la ponctuation ?": 0})
wb.record("B4.Q8b", 1, mistakes={"relis la définition de « contient la réponse » : on cherche la référence dans la réponse": 0})
wb.record("B4.Q8c", Q8C, decimals=3, mistakes={"c'est la variance : on demande l'écart-type": 0.0016,
          "relis la formule de l'écart-type d'une proportion (§204.5) : n est-il sous la racine ?": 0.004,
          "relis la formule : n = 100 questions": 0.4})
wb.record("B4.Q8d", "C", mistakes={"relis « Le juge LLM » : un petit modèle fait-il un bon juge ?": "A", "l'échelle de la note change-t-elle le biais de position ?": "B"})
wb.record("B4.Q8e", True, mistakes={"relis « La contamination » : quel indice la fiche donne-t-elle ?": False})

WB_ANSWER {"hash": "833e894b26786a0088f301b383d11edbaf0973044f422f1fbacd000ffa40e421", "id": "B4.Q8a", "kind": "int", "magnitude": 0, "mistakes": {"48407692161b10d5eb73f82f3485d2d2720156a8098a7e122291a83a428677a2": "relis la normalisation de l'exact match : que deviennent la casse et la ponctuation ?"}, "sign": 1}
📝 Ex B4.Q8a : réponse enregistrée (int).
WB_ANSWER {"hash": "449942d9aff865c7579e0ffcc5f5368b9be7ab0ca3283ce5bebd6bedc5baf82b", "id": "B4.Q8b", "kind": "int", "magnitude": 0, "mistakes": {"ff2af04ebf5aab19a6b1a2621d35fa61a571c8d82b677f03fbe531b37c385cae": "relis la définition de « contient la réponse » : on cherche la référence dans la réponse"}, "sign": 1}
📝 Ex B4.Q8b : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "a6cc002c39acc3de515aea12aed6ccb049407053470b6e74f2852b3004a40293", "hash_coarse": "75973d4a32053e114c9adc62e4b0574350e9f3c03c72ea80f269ca97a8e68075", "id": "B4.Q8c", "kind": "float", "magnitude": -2, "mistakes": {"5e8d195b0c2d2e704c2eebea05259379a1

**B4.Q9 — BM25 ou recherche dense : qui retrouve « Watson » et qui retrouve « le docteur » ?**

In [11]:
wb.record("B4.Q9a", "B", mistakes={"l'idf pondère les mots communs : BM25 connaît-il des synonymes ?": "A", "le passage 0 contient-il le mot de la requête ?": "C"})
wb.record("B4.Q9b", True, mistakes={"relis la première des quatre idées de BM25 (§204.6)": False})
wb.record("B4.Q9c", "B", mistakes={"la recherche dense utilise-t-elle BM25 ? Relis « La recherche dense »": "A", "un embedding compte-t-il des lettres ? Relis « La recherche dense »": "C"})
wb.record("B4.Q9d", "A", mistakes={"relis le paragraphe sur les forces des deux recherches : un embedding connaît-il bien un code ?": "B", "relis le paragraphe sur les forces des deux recherches : se valent-elles sur un code ?": "C"})
wb.record("B4.Q9e", Q9E, decimals=4, mistakes={"relis la formule de la RRF : les rangs partent de 1, et k s'ajoute au rang": 2 / (60 + 2),
          "relis la question : le document est dans deux classements": 1 / 63,
          "relis la formule de la RRF : k = 60 s'ajoute au rang": 2 / 3})

WB_ANSWER {"hash": "ea82ae48811774870820fef58aad135af773a05d4a177cd52e8f8fcac7f24fcd", "id": "B4.Q9a", "kind": "str", "mistakes": {"7522d424975c036968a80f5f700f76b9617e63cf4912c80c44089963271372bc": "l'idf pondère les mots communs : BM25 connaît-il des synonymes ?", "bcea2eec3f0d1fe148075ad6341238e7c0ff54d3ccc6a6170fe441663aca2501": "le passage 0 contient-il le mot de la requête ?"}}
📝 Ex B4.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "c2e9ebba28a57a943bc5e76cfd0e7523aec6652e405f1371b95c96a83a67b13c", "id": "B4.Q9b", "kind": "bool", "mistakes": {"891b3eee8b6abe94dfdc3bbd947bcecab3474a12cb03a2925e1cb3131c3fe7e2": "relis la première des quatre idées de BM25 (§204.6)"}}
📝 Ex B4.Q9b : réponse enregistrée (bool).
WB_ANSWER {"hash": "0a0983374d623854051a60a89abf26c92cc5e940e84516ad27e4c3ee164ca2c6", "id": "B4.Q9c", "kind": "str", "mistakes": {"3b961878071bd2e1efda824c16c68a003cb58d44d612290223acd8f3f717ec51": "la recherche dense utilise-t-elle BM25 ? Relis « La recherche dense »", "b

**B4.Q10 — RAG ou fine-tuning pour ajouter des connaissances ?**

In [12]:
wb.record("B4.Q10a", "B", mistakes={"relis « RAG ou fine-tuning ? » : un fine-tuning apprend-il bien des faits précis ?": "A", "relis « RAG ou fine-tuning ? » : que demande une documentation qui change chaque semaine ?": "C"})
wb.record("B4.Q10b", "B", mistakes={"relis « RAG ou fine-tuning ? » : le RAG change-t-il un comportement ?": "A", "relis « RAG ou fine-tuning ? » : à quoi sert le fine-tuning ?": "C"})
wb.record("B4.Q10c", True, mistakes={"relis les arguments du RAG dans « RAG ou fine-tuning ? »": False})
wb.record("B4.Q10d", False, mistakes={"relis « RAG ou fine-tuning ? » : un fine-tuning sur quelques exemples apprend-il bien des faits ?": True})
wb.record("B4.Q10e", "A", mistakes={"relis l'étape 1 des quatre étapes du RAG : qu'est-ce qui se calcule une fois ?": "B", "l'index contient-il encore l'ancienne procédure ?": "C"})

WB_ANSWER {"hash": "e2ece298b1846a755f627fbed86082e4814368d71d92a65b3c8f1388e48c4d51", "id": "B4.Q10a", "kind": "str", "mistakes": {"490fd5ebbfdab029d6948f5bc486d049a2448d680084653f53ee62caebabf46a": "relis « RAG ou fine-tuning ? » : un fine-tuning apprend-il bien des faits précis ?", "cae6c813d3a33584e32a96b50e4ae7b8f0406ad35c5088f5639a0c20c73fc9ce": "relis « RAG ou fine-tuning ? » : que demande une documentation qui change chaque semaine ?"}}
📝 Ex B4.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "4e796e660da0b987b125fcd6e9c7dee0ad4308a411cdf45d75a0a716fabb5fbd", "id": "B4.Q10b", "kind": "str", "mistakes": {"132298364ca1178b4185ad734e3be0974d591ad3ed5eac24d3b7e6d666381804": "relis « RAG ou fine-tuning ? » : à quoi sert le fine-tuning ?", "6c55e574386abb3f3434ba6a3ac7921efe4a0957f85253d29c6346c53a103d31": "relis « RAG ou fine-tuning ? » : le RAG change-t-il un comportement ?"}}
📝 Ex B4.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "c9a147a1478b588bf028f54d8052bada9bd9c7e2

**B4.Q11 — LoRA : ce qui est gelé, ce qui est appris, ce que l'on fusionne**

In [13]:
wb.record("B4.Q11a", "B", mistakes={"relis la première puce de « Ce que dit chaque morceau » (§204.8)": "A", "relis la formule de LoRA : W disparaît-elle ?": "C"})
wb.record("B4.Q11b", True, mistakes={"relis la deuxième puce de « Ce que dit chaque morceau » : que vaut BA au départ ?": False})
wb.record("B4.Q11c", Q11C, mistakes={"c'est le nombre de poids de la matrice : on demande ceux de LoRA": 512 * 512,
          "relis la formule r(d + k) : il y a deux matrices, A et B": 4 * 512,
          "relis la question : r = 4": 8 * 1024})
wb.record("B4.Q11d", "C", mistakes={"relis « Fusionner l'adaptateur » : que deviennent les branches ?": "A", "relis « Fusionner l'adaptateur » : que devient W ?": "B"})
wb.record("B4.Q11e", False, mistakes={"calcule le facteur α/r avec les valeurs de la question": True})

WB_ANSWER {"hash": "f347a669c2ca526262f8c31fb43b946a1b1b711d15084e7d6e13e4f6c2e13812", "id": "B4.Q11a", "kind": "str", "mistakes": {"60768e19c59b2d914df857d41e316fff958a97203fdd66e3ebf866742870837b": "relis la formule de LoRA : W disparaît-elle ?", "e257d7034c6f6e7c2cfbcae87260402f35eaa45e3a057f3fa9b048416a1c1002": "relis la première puce de « Ce que dit chaque morceau » (§204.8)"}}
📝 Ex B4.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "3e13bc7d7988716230f7ad829a9285cf4bef00f54b17e13a80363047c7f156c1", "id": "B4.Q11b", "kind": "bool", "mistakes": {"aaaaeb97260df8b57221c96c2afe17d81a60f11752778bf530524ec8c4c7b8b7": "relis la deuxième puce de « Ce que dit chaque morceau » : que vaut BA au départ ?"}}
📝 Ex B4.Q11b : réponse enregistrée (bool).
WB_ANSWER {"hash": "960ae16d13ae74585da02a6791323087b012dde0ee4f535d8a3f60a92c0ba990", "id": "B4.Q11c", "kind": "int", "magnitude": 3, "mistakes": {"a2ba28ad62de38555c3ab23c561c1e1e4ba3b005e8c88bf6b3361e48d6008ebd": "relis la question : r = 4

**B4.Q12 — Injection de prompt : quand un document récupéré devient une instruction**

In [14]:
wb.record("B4.Q12a", "B", mistakes={"c'est l'injection directe : relis « L'injection de prompt » (§204.10)": "A", "c'est une hallucination : relis la définition de l'injection": "C"})
wb.record("B4.Q12b", False, mistakes={"relis « Les parades réduisent le risque sans l'annuler » (§204.10)": True})
wb.record("B4.Q12c", "C", mistakes={"une température basse empêche-t-elle le modèle de suivre une instruction ?": "A", "relis les parades de l'injection : la taille du modèle en fait-elle partie ?": "B"})
wb.record("B4.Q12d", True, mistakes={"relis la définition de l'injection indirecte : d'où viennent les instructions ?": False})
wb.record("B4.Q12e", "A", mistakes={"relis l'encadré 🕰️ sur l'OWASP : à quel rang est l'agentivité excessive ?": "B", "relis l'encadré 🕰️ sur l'OWASP : à quel rang est la désinformation ?": "C"})

WB_ANSWER {"hash": "b1cd0c6f11566641b4c119b75451ef7c349700d0d44bd8e798b92758bbcc0dcd", "id": "B4.Q12a", "kind": "str", "mistakes": {"6d3f5bf1ac6738c149f60bee5030459f24ad5d72c2f52b7daea4e23f348dfeb1": "c'est l'injection directe : relis « L'injection de prompt » (§204.10)", "a07bf8f4f8e9a38ed54ac337b88873675f0b061f11e11a29c3e373cd85fe53fd": "c'est une hallucination : relis la définition de l'injection"}}
📝 Ex B4.Q12a : réponse enregistrée (str).
WB_ANSWER {"hash": "5f083a173ef2345ec0f574e199a6345e1b43f94d49e66264ce5ac46519de55ef", "id": "B4.Q12b", "kind": "bool", "mistakes": {"f2a7b84702d28ee4d22a9ccedcaca795505c10b71a44b214f081346b7fb066dd": "relis « Les parades réduisent le risque sans l'annuler » (§204.10)"}}
📝 Ex B4.Q12b : réponse enregistrée (bool).
WB_ANSWER {"hash": "3ca252ab5da599a71b4c19329162a028ba10f8129fb94f1f3567b6cdd6b094cf", "id": "B4.Q12c", "kind": "str", "mistakes": {"13bd82fc1e04bff65d9800f25e753d3f5133a56435b309c375c8df92156e68ff": "relis les parades de l'injection : l

**B4.R1 — Ce que le masque causal et le KV cache imposent à la génération**

In [15]:
wb.record("B4.R1a", 100, mistakes={"avec un KV cache, le prompt repasse-t-il dans le modèle à chaque token ?": 500,
          "relis la question : la passe sur le prompt compte, et elle donne déjà le premier token": 99,
          "la passe sur le prompt produit-elle déjà un token ?": 101,
          "relis la question : on compte des passes avant, une par token produit": 400})
wb.record("B4.R1b", False, mistakes={"relis le 🔁 Rappel B3 (§204.3) : peut-on écrire le token 2 avant de connaître le token 1 ?": True})
wb.record("B4.R1c", R1C, mistakes={"n'oublie pas le facteur 2 des clés et des valeurs": R1C // 2,
          "combien d'octets par valeur en bfloat16 ?": R1C * 2,
          "relis la formule du KV cache (§204.2) : ce sont les têtes de clés et valeurs qui comptent (GQA)": 2 * 24 * 896 * 2})
wb.record("B4.R1d", "C", mistakes={"le nombre de têtes change-t-il le nombre de passes avant ?": "A", "relis le 🔁 Rappel B3 de §204.3 : de quoi le temps d'une réponse est-il proportionnel ?": "B"})
wb.record("B4.R1e", True, mistakes={"relis le masque causal (B3) : le token 50 voit-il le token 49 ?": False})

WB_ANSWER {"hash": "067115c3433a68e94721c85b4a37251ea3a2bd691cf4831b30b95b1396e3650b", "id": "B4.R1a", "kind": "int", "magnitude": 2, "mistakes": {"6bd47ceb0cf68d6ba930a426dbfe2dd308ff84f272ce03c7c30b9aebd2a708d4": "relis la question : la passe sur le prompt compte, et elle donne déjà le premier token", "84bde6e42977503d626bcdbba16ce401f47d09b5151507347af526f336565fef": "avec un KV cache, le prompt repasse-t-il dans le modèle à chaque token ?", "8b8f4791d91029b2921095acb79d72c9864e557705dbf3e7be37e9962d950e93": "relis la question : on compte des passes avant, une par token produit", "a8b5043a98ce944d1226eed47d1c9334cfbd4e229c895e1126b278ed518b2932": "la passe sur le prompt produit-elle déjà un token ?"}, "sign": 1}
📝 Ex B4.R1a : réponse enregistrée (int).
WB_ANSWER {"hash": "e0dfd307da50a71ba5f86517130aa9f95876328885107f217415bcd5b9e5e584", "id": "B4.R1b", "kind": "bool", "mistakes": {"a47f86becd6f10845ef05b3dcf54b60e9294dd5f6afc06b0d55c1cc08514758a": "relis le 🔁 Rappel B3 (§204.3) : p

**B4.R2 — Extraction de features ou fine-tuning : que change le nombre d'exemples ?**

In [16]:
wb.record("B4.R2a", "B", mistakes={"relis le 🔁 Rappel B1 (§204.8) : avec peu d'exemples, qui gagnait ?": "A", "relis le 🔁 Rappel B1 (§204.8) : les deux méthodes donnaient-elles la même chose ?": "C"})
wb.record("B4.R2b", True, mistakes={"relis B1 : que gèle l'extraction de features ?": False})
wb.record("B4.R2c", "A", mistakes={"relis le 🔁 Rappel B1 (§204.8) : où LoRA place-t-il ses paramètres ?": "B", "relis la première puce de « Ce que dit chaque morceau » : W est-elle entraînée ?": "C"})
wb.record("B4.R2d", False, mistakes={"relis le 🔁 Rappel B1 (§204.8) : que risque un fine-tuning complet avec peu d'exemples ?": True})
wb.record("B4.R2e", "A", mistakes={"c'est un problème de contexte, pas d'entraînement : relis B1": "B", "des poids sauvegardés se perdent-ils ? Relis la définition de B1": "C"})

WB_ANSWER {"hash": "9214972f214ed47923931ab3953c5eeee53e085e58cd01b7460d1fa3e53952a2", "id": "B4.R2a", "kind": "str", "mistakes": {"8ccecf20ebc6d7e39668c975f3c6cced15c8adac6850dd7538059abd2b47aa07": "relis le 🔁 Rappel B1 (§204.8) : avec peu d'exemples, qui gagnait ?", "b8d5339418a1e8fd6c66ee4a467acfeeefcf391636d808f1d24fb6a453f67175": "relis le 🔁 Rappel B1 (§204.8) : les deux méthodes donnaient-elles la même chose ?"}}
📝 Ex B4.R2a : réponse enregistrée (str).
WB_ANSWER {"hash": "bb6a1c3ed901efab37b627a069f4a1778b5d413892d2d5f59aa79abd9a234243", "id": "B4.R2b", "kind": "bool", "mistakes": {"50909381c6dfd2b4aac7e9591cbea4cf1e4c9bf2e18a32a294a12086a670414d": "relis B1 : que gèle l'extraction de features ?"}}
📝 Ex B4.R2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "2b842390b13ca61d6518e39572687820fcd80ac2e4d17bb162b4739e8da35dac", "id": "B4.R2c", "kind": "str", "mistakes": {"3737712df38fcdb9e6aa6ec76bb71223be5753374d72535943c8c45dc4a207d9": "relis la première puce de « Ce que dit chaq

**B4.R3 — Les k plus proches voisins et le choix de la distance**

In [17]:
wb.record("B4.R3a", COS_R3, decimals=2, mistakes={"c'est le produit scalaire : divise-le par les normes": 50.0,
          "c'est la distance euclidienne : on demande le cosinus": 5.0})
wb.record("B4.R3b", DIST_R3, decimals=2, mistakes={"c'est le carré de la distance : prends sa racine": 25.0,
          "c'est le cosinus : on demande la distance euclidienne": 1.0,
          "la distance euclidienne est la racine de la somme des carrés des écarts": 7.0})
wb.record("B4.R3c", SQ_R3, decimals=2, mistakes={"relis la formule de la fiche (§204.6) : il y a un facteur 2": 1 - 0.82,
          "c'est la distance : on demande son carré": math.sqrt(SQ_R3),
          "vérifie le signe dans la formule de la fiche": 2 + 2 * 0.82})
wb.record("B4.R3d", True, mistakes={"relis la formule qui relie distance et cosinus pour des vecteurs de norme 1 (§204.6)": False})
wb.record("B4.R3e", MULT_R3, mistakes={"un produit scalaire de dimension 768 demande 768 multiplications : relis la question": 50_000,
          "relis la question : la dimension est 768": 50_000 * 384,
          "relis la question : 50 000 chunks": 10_000 * 768})

WB_ANSWER {"decimals": 2, "hash": "f379ac27a21ad2f2f3349febafe2117cf5d63e999692e2a1ee3ddefb661f3e3c", "hash_coarse": "f2afb8a8b69a7a5f714592514810db6e90636a0bf6ebef568848769a182933be", "id": "B4.R3a", "kind": "float", "magnitude": 0, "mistakes": {"0239ca320eda15398f9e57be389d667e2b0d687c543cf7354fa30953d343ae92": "c'est la distance euclidienne : on demande le cosinus", "9ce4ceff2e31214609337c978940392b7d133573ca49b72af9365414fc549c86": "c'est le produit scalaire : divise-le par les normes"}, "sign": 1}
📝 Ex B4.R3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "6895356937758f1cfd78bbeb386c77c4f72b2183ad1788771980c75c03a7237e", "hash_coarse": "bdbec5505bc2cb5cad3273bf2c43ffd633f93f03ce3fb90141c59f8cb037ed08", "id": "B4.R3b", "kind": "float", "magnitude": 0, "mistakes": {"8b4fd5e221b3812847353819fb92fcf267275f9b3e451f378f5f4006beff7722": "c'est le carré de la distance : prends sa racine", "c49d254ba223ab38ff1770a2ec794028a8809a1b870f286d99fd6d2c34217680":

**Ex B4.1 — Mémoire d'un modèle : paramètres, dtype et KV cache — tient-il sur ton ordinateur ?**

In [18]:
wb.record("B4.1a", EMB_B41, mistakes={"les poids liés ne comptent qu'une fois : relis l'énoncé": 2 * EMB_B41,
          "la table a une ligne par token du vocabulaire, de taille d": V_B41 * 64,
          "relis l'énoncé : V = 49 152 tokens": 50_257 * D_B41})
wb.record("B4.1b", EMB_SHARE_B41, decimals=1, mistakes={"on demande un pourcentage, pas une proportion": EMB_SHARE_B41 / 100,
          "les poids liés ne comptent qu'une fois : relis l'énoncé": 2 * EMB_SHARE_B41})
wb.record("B4.1c", FP32_B41, decimals=1, mistakes={"combien d'octets par paramètre en float32 ?": BF16_B41,
          "1 Mo = 10⁶ octets : vérifie la conversion": 4 * N_B41 / 2 ** 20,
          "c'est le nombre de bits : on demande des octets": 32 * N_B41 / MB})
wb.record("B4.1d", BF16_B41, decimals=1, mistakes={"combien d'octets par paramètre en bfloat16 ?": FP32_B41,
          "1 Mo = 10⁶ octets : vérifie la conversion": 2 * N_B41 / 2 ** 20,
          "relis la question : bfloat16, pas int4": INT4_B41})
wb.record("B4.1e", INT4_B41, decimals=1, mistakes={"combien d'octets par paramètre en int4 ? Relis l'énoncé": 4 * N_B41 / MB,
          "relis l'énoncé : 0,5 octet par paramètre": N_B41 / MB,
          "1 Mo = 10⁶ octets : vérifie la conversion": 0.5 * N_B41 / 2 ** 20})
wb.record("B4.1f", KV_TOKEN_B41, mistakes={"n'oublie pas le facteur 2 des clés et des valeurs": KV_TOKEN_B41 // 2,
          "combien d'octets par valeur en bfloat16 ?": KV_TOKEN_B41 // 2 * 4,
          "relis la formule du KV cache : ce sont les têtes de clés et valeurs (GQA) qui comptent": 2 * 30 * 9 * 64 * 2,
          "relis la formule du KV cache : une couche, ou toutes ?": KV_TOKEN_B41 // 30})
wb.record("B4.1g", KV_FULL_B41, decimals=1, mistakes={"relis la formule du KV cache : ce sont les têtes de clés et valeurs (GQA) qui comptent": 3 * KV_FULL_B41,
          "n'oublie pas le facteur 2 des clés et des valeurs": KV_FULL_B41 / 2,
          "1 Mo = 10⁶ octets : vérifie la conversion": KV_TOKEN_B41 * CTX_B41 / 2 ** 20,
          "relis l'énoncé : le contexte compte 8 192 tokens": KV_TOKEN_B41 * 8_000 / MB})
wb.record("B4.1h", False, mistakes={"compare tes réponses à g) et d)": True})
wb.record("B4.1i", CONVERSATIONS_B41, mistakes={"n'oublie pas de retirer les poids des 2 Go": int(2 * GB // (KV_TOKEN_B41 * CTX_B41)),
          "relis la question : les poids sont en bfloat16": int((2 * GB - 4 * N_B41) // (KV_TOKEN_B41 * CTX_B41))},
          fractional="on demande un nombre de conversations entières")
wb.record("B4.1j", round(SPEED_B41), mistakes={"c'est la vitesse de Qwen2.5 : applique la proportion des mémoires": round(QWEN_TOKENS / QWEN_SECONDS),
          "un modèle plus petit va-t-il plus vite ou plus lentement ? Vérifie le sens du rapport": round(QWEN_TOKENS / QWEN_SECONDS * 2 * N_B41 / QWEN_FOOTPRINT),
          "relis la question : compare les mémoires en bfloat16 des deux modèles": round(QWEN_TOKENS / QWEN_SECONDS * QWEN_FOOTPRINT / (4 * N_B41))},
          fractional="arrondis à l'entier le plus proche")
wb.record("B4.1k", True, mistakes={"compare ta réponse à c) à 1 Go = 1 000 Mo": False})

WB_ANSWER {"hash": "f4ed400be076d787cbf83579ac7940018ea93cb7a7b6eb6ab20afc1a102ee11a", "id": "B4.1a", "kind": "int", "magnitude": 7, "mistakes": {"16714fab10fd944ed03cc91a475ddae097e7d2306d8a6842dd115a118b534e95": "la table a une ligne par token du vocabulaire, de taille d", "823aba6cb1d43ebf824fc61717a0c727290bc00ad8a1180d43c4d2b3a46e2425": "relis l'énoncé : V = 49 152 tokens", "8d90800a2523f2473033369a6b0850afa58a1d9d46a6b5b6fc5a4abaea38c487": "les poids liés ne comptent qu'une fois : relis l'énoncé"}, "sign": 1}
📝 Ex B4.1a : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "4d756568c10eb39226a23fd0a564f18e56a2a5edc0171363d953fc44f82b0eba", "hash_coarse": "85fbf3b71558fb9537daacbd067438ec144eeac0f36e9a887c2ff0291851843b", "id": "B4.1b", "kind": "float", "magnitude": 1, "mistakes": {"5c12fb7ac0fd09b5fee3b39336707ff63bd8fc5146b441543aaaaaab0b886f56": "on demande un pourcentage, pas une proportion", "9da1d0944908b41df4977b8330b28fd424a5978486c6c288518c53d643050077": "les poi

**Ex B4.2 — Recall@k et MRR pour cinq requêtes**

In [19]:
wb.record("B4.2a", recall_b42(1), decimals=3, mistakes={"relis la définition du recall@k : par quoi divise-t-on, pour chaque question ?": 0.2})
wb.record("B4.2b", recall_b42(3), decimals=3, mistakes={"le recall divise par le nombre de pertinents, pas par k": recall_b42(3, denominator="k"),
          "relis la question : on demande le recall, pas la part des questions réussies": HIT3_B42,
          "relis la question : les trois premiers résultats": recall_b42(2)})
wb.record("B4.2c", recall_b42(5), decimals=3, mistakes={"le recall divise par le nombre de pertinents, pas par k": recall_b42(5, denominator="k"),
          "relis la question : on demande le recall, pas la part des questions réussies": 0.8,
          "relis la question : les cinq premiers résultats": recall_b42(4)})
wb.record("B4.2d", MRR_B42, decimals=3, mistakes={"on prend le rang du premier pertinent, pas la somme sur tous les pertinents": (1 / 2 + 1 / 3 + 1 / 5 + 1 + 0 + 1 / 3 + 1 / 4) / 5,
          "une question sans pertinent retrouvé compte 0, et la moyenne porte sur les cinq questions": MRR_B42 * 5 / 4,
          "le MRR moyenne l'inverse du rang, pas le rang": (2 + 3 + 1 + 0 + 3) / 5})
wb.record("B4.2e", HIT3_B42, decimals=3, mistakes={"relis la question : on compte les questions qui ont au moins un pertinent, pas les pertinents": recall_b42(3),
          "relis la question : les trois premiers résultats": 0.4})
wb.record("B4.2f", False, mistakes={"compare tes réponses à b) et e)": True})
wb.record("B4.2g", FULL5_B42, mistakes={"relis la question : tous les pertinents de la question, pas au moins un": 4,
          "recompte question par question : une question à deux pertinents peut aussi avoir un recall@5 de 1": 1,
          "recompte question par question : a-t-on retrouvé tous les pertinents de chacune ?": 3})
wb.record("B4.2h", True, mistakes={"pour une question, compare la part de ses pertinents au rang 1 et l'inverse du rang de son premier pertinent": False})

WB_ANSWER {"decimals": 3, "hash": "274b6bc6de6218d9c5aaaf2b7f2f5b18d8e24aeb27d0bce1d4dc871e06a5d0b8", "hash_coarse": "4caf36ea45e22a0a2bae249942f7cc92af5a446badf84873eae205e671df66ef", "id": "B4.2a", "kind": "float", "magnitude": -1, "mistakes": {"114e11ac6818f21d49ab3f60b07b576834a28c34438dd70d232277044371dee2": "relis la définition du recall@k : par quoi divise-t-on, pour chaque question ?"}, "sign": 1}
📝 Ex B4.2a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "319b38f8711906276f631a0f45dbcc12429b079fb817e1e7f7bb774d21cc4668", "hash_coarse": "714a55b1c12a927abdee2c62f2d53fd854037319b20b62155cd84b091a7b3b33", "id": "B4.2b", "kind": "float", "magnitude": -1, "mistakes": {"8f1709558b44b6aa0eb044093ea5dc265f4f9f608ff0178cd96b8dff39ae1993": "relis la question : on demande le recall, pas la part des questions réussies", "9de21c04907f530dd336cac77a167a22a805a952d1ab0a2f16fa2e1ad96a9b3c": "relis la question : les trois premiers résultats", "a1c00225cff981790f

**Ex B4.3 — LoRA : paramètres entraînables d'une couche 896×896 de rang 8, puis de tout un modèle**

In [20]:
wb.record("B4.3a", Q_B43, mistakes={"relis la formule r(d + k) : il y a deux matrices, A et B": R_B43 * D_QWEN,
          "c'est le nombre de poids de la matrice : on demande ceux de LoRA": D_QWEN ** 2,
          "relis la formule r(d + k) : r s'ajoute-t-il ou se multiplie-t-il ?": 2 * D_QWEN + R_B43})
wb.record("B4.3b", Q_SHARE_B43, decimals=2, mistakes={"on demande un pourcentage, pas une proportion": Q_SHARE_B43 / 100,
          "relis la question : la matrice de q_proj, sans son biais": 100 * Q_B43 / (D_QWEN ** 2 + D_QWEN)})
wb.record("B4.3c", K_SHARE_B43, decimals=2, mistakes={"relis l'énoncé : k_proj va de 896 à 128": Q_SHARE_B43,
          "on demande un pourcentage, pas une proportion": K_SHARE_B43 / 100})
wb.record("B4.3d", LAYER_B43, mistakes={"relis l'énoncé : k_proj et v_proj vont de 896 à 128": 4 * Q_B43,
          "relis la question : quatre matrices, q, k, v et o": 2 * Q_B43 + K_B43,
          "relis la question : une seule couche": MODEL_B43})
wb.record("B4.3e", MODEL_B43, mistakes={"relis l'énoncé : k_proj et v_proj vont de 896 à 128": LAYERS_QWEN * 4 * Q_B43,
          "relis l'énoncé : 24 couches": 28 * LAYER_B43,
          "relis la question : q, k, v et o, pas seulement q et v": LAYERS_QWEN * (Q_B43 + K_B43)})
wb.record("B4.3f", TRAINABLE_PCT_B43, decimals=4, mistakes={"relis la question : « all params » compte aussi les adaptateurs": 100 * MODEL_B43 / N_QWEN,
          "on demande un pourcentage, pas une proportion": TRAINABLE_PCT_B43 / 100})
wb.record("B4.3g", ALL_LINEAR_B43, mistakes={"relis l'énoncé : les trois couches du MLP comptent aussi": MODEL_B43,
          "relis l'énoncé : le MLP va de 896 à 4 864, puis revient à 896": LAYERS_QWEN * (LAYER_B43 + 3 * Q_B43),
          "relis la question : sans la tête de sortie": ALL_LINEAR_B43 + lora_params(151_936, D_QWEN)})
wb.record("B4.3h", True, mistakes={"relis la formule r(d + k) : comment dépend-elle de r ?": False})
wb.record("B4.3i", ADAPTER_MB_B43, decimals=1, mistakes={"combien d'octets par paramètre en float32 ?": 2 * MODEL_B43 / MB,
          "1 Mo = 10⁶ octets : vérifie la conversion": 4 * MODEL_B43 / 2 ** 20,
          "c'est la mémoire du modèle entier : on demande celle de l'adaptateur": 4 * (N_QWEN + MODEL_B43) / MB})
wb.record("B4.3j", NO_SAVING_RANK_B43, mistakes={"relis la question : « au moins autant » ; vérifie le cas d'égalité": NO_SAVING_RANK_B43 - 1,
          "relis la formule r(d + k) : il y a deux matrices, A et B": D_QWEN,
          "écris l'inégalité r(d + k) ≥ dk et résous-la en r": NO_SAVING_RANK_B43 + 1},
          fractional="on demande un rang, un nombre entier")
wb.record("B4.3k", False, mistakes={"compare tes réponses à b) et c)": True})

WB_ANSWER {"hash": "226cf822e0985a8ac70ab4b830b82327a2ee9cb82995de9c44fe340f775d4ace", "id": "B4.3a", "kind": "int", "magnitude": 4, "mistakes": {"12df512073673d46bf0c7e9a925cf0a927a2b92df6f87c408e96bb6fe78a8227": "relis la formule r(d + k) : r s'ajoute-t-il ou se multiplie-t-il ?", "26f06cc1b71324ae8d48c3f6f87ddff71c162d5805e13ed77f500edb1a536a60": "relis la formule r(d + k) : il y a deux matrices, A et B", "a9b0b59969bfaa4f9083fd87c541c3cfcdf824998b2ef335d49f459628372215": "c'est le nombre de poids de la matrice : on demande ceux de LoRA"}, "sign": 1}
📝 Ex B4.3a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "b12ce31c70accc2388d63a737df8fe200d8515dc48180447d505a35bfcbeff71", "hash_coarse": "2b49426c08635d20a6a2b1af32e0707f9d5df7dfa170e9f5f7210e5db137e425", "id": "B4.3b", "kind": "float", "magnitude": 0, "mistakes": {"329b4815d63459479322388321d5d95b68ddf5e1f4ec8ae2f1e3bc1c8e5e6a7b": "relis la question : la matrice de q_proj, sans son biais", "6eabb9e26e655b0bb9c373a1e

**Ex B4.4 — Un chat template à la main : rôles, tokens spéciaux et prompt de génération**

In [21]:
wb.record("B4.4a", U_GEN_B44[1], mistakes={"relis la condition `if loop.first and …` du gabarit : que se passe-t-il sans message système ?": U_GEN_B44[1] - 2,
          "relis la question : add_generation_prompt=True": U_GEN_B44[1] - 1,
          "les retours à la ligne ne sont pas des tokens spéciaux : relis le tableau": U_GEN_B44[1] + 5})
wb.record("B4.4b", U_GEN_B44[2], mistakes={"relis la condition `if loop.first and …` du gabarit : que se passe-t-il sans message système ?": U_GEN_B44[2] - 21,
          "relis le tableau : `assistant` compte 2 tokens": U_GEN_B44[2] - 1,
          "n'oublie pas les retours à la ligne, un token chacun": U_GEN_B44[2] - 5,
          "relis la question : add_generation_prompt=True": U_GEN_B44[2] - 4})
wb.record("B4.4c", SU_GEN_B44[2], mistakes={"relis la condition `if loop.first and …` : le message système par défaut est-il ajouté quand S existe ?": DOUBLE_SYSTEM_B44[2],
          "relis le tableau : `assistant` compte 2 tokens": SU_GEN_B44[2] - 1,
          "n'oublie pas les retours à la ligne, un token chacun": SU_GEN_B44[2] - 5,
          "relis la question : add_generation_prompt=True": SU_NOGEN_B44[2]})
wb.record("B4.4d", SU_NOGEN_B44[2], mistakes={"relis la question : add_generation_prompt=False": SU_GEN_B44[2],
          "n'oublie pas les retours à la ligne, un token chacun": SU_NOGEN_B44[2] - 4,
          "relis la condition `if loop.first and …` : le message système par défaut est-il ajouté quand S existe ?": DOUBLE_SYSTEM_B44[2] - 4})
wb.record("B4.4e", TURN2_B44[2], mistakes={"relis la question : le prompt du deuxième tour contient toute la conversation": SU_GEN_B44[2],
          "relis le tableau : `assistant` compte 2 tokens, aussi dans le tour de A": TURN2_B44[2] - 1,
          "le tour de A est-il fermé par `<|im_end|>\n` comme les autres ?": TURN2_B44[2] - 2,
          "relis la condition `if loop.first and …` : le message système par défaut s'ajoute-t-il ici ?": TURN2_B44[2] + 21})
wb.record("B4.4f", "B", mistakes={"relis la condition `if loop.first and …` : le message système par défaut est-il ajouté quand S existe ?": "A", "relis la question : add_generation_prompt=True": "C"})
wb.record("B4.4g", False, mistakes={"relis le tableau : quels morceaux sont des tokens spéciaux ? En vois-tu un dans cette chaîne ?": True})
wb.record("B4.4h", True, mistakes={"relis le gabarit : la boucle `for` parcourt-elle tous les messages ?": False})

WB_ANSWER {"hash": "ce9f7b731728127c9c6bc5e7762469721a9ef0a213b6eebad2a1c090d1d51329", "id": "B4.4a", "kind": "int", "magnitude": 0, "mistakes": {"3de714dcc1b9e73ac77d8c5b4ab6659defdd00574cbd779aec04152a75cab657": "les retours à la ligne ne sont pas des tokens spéciaux : relis le tableau", "91389aaf537326ad6f719b90376a572af65cd85d7fe29138090423b98cedd2bb": "relis la question : add_generation_prompt=True", "aefbd1c32da7a560522f901b0e24ffb3355e854641a73b9aa1bac3f0fda28305": "relis la condition `if loop.first and …` du gabarit : que se passe-t-il sans message système ?"}, "sign": 1}
📝 Ex B4.4a : réponse enregistrée (int).
WB_ANSWER {"hash": "7c55b83e7df37efc2fc2921cb22ad8ae841cd54f4d7f230bb0d20c38b8d36bba", "id": "B4.4b", "kind": "int", "magnitude": 1, "mistakes": {"24969b1bc3c3a04b94d5f176761ccc4df654ffffde921d08c3de63b2becb7ce7": "relis la condition `if loop.first and …` du gabarit : que se passe-t-il sans message système ?", "3412fb181be4c795518114591e5a6af34bcf98d857f53a0c7d2407840165

**Ex B4.5 — BM25 à la main : trois documents, une requête de deux mots**

In [22]:
wb.record("B4.5a", AVGDL_B45, decimals=4, mistakes={"compte à nouveau les mots de chaque document": 16 / 3,
          "relis la question : la longueur moyenne, pas totale": 15.0})
wb.record("B4.5b", IDF_NAUTILUS_B45, decimals=4, mistakes={"relis la formule : l'idf de la docstring a un « 1 + »": math.log(0.5 / 3.5),
          "n_t compte les documents qui contiennent le terme, pas ses occurrences": math.log(1 + (3 - 4 + 0.5) / (4 + 0.5)),
          "relis la formule : n_t + 0,5 au dénominateur": math.log(1 + 0.5 / 3),
          "le logarithme est naturel ici, pas décimal": math.log10(1 + 0.5 / 3.5)})
wb.record("B4.5c", SCORES_B45[1], decimals=4, mistakes={"relis la formule : le dénominateur dépend de la longueur relative du document": float(SCORES_B0_B45[1]),
          "relis la formule : le facteur k₁ + 1 au numérateur": float(SCORES_B45[1] / 2.5),
          "n_t compte les documents qui contiennent le terme, pas ses occurrences": float(SCORES_B45[1] / IDF_NAUTILUS_B45 * math.log(1 + (3 - 4 + 0.5) / 4.5))})
wb.record("B4.5d", SCORES_B45[2], decimals=4, mistakes={"le doc 2 contient « nautilus » deux fois : f = 2": float(bm25_b45(docs=[DOCS_B45[0], DOCS_B45[1], ["le", "nautilus", "croise", "un", "x", "au", "grand", "large"]])[2]),
          "relis la formule : le dénominateur dépend de la longueur relative du document": float(SCORES_B0_B45[2]),
          "relis la formule : le facteur k₁ + 1 au numérateur": float(SCORES_B45[2] / 2.5)})
wb.record("B4.5e", SCORES_B45[0], decimals=4, mistakes={"n'oublie pas la contribution de « nautilus »": float(bm25_b45(query=["nemo"])[0]),
          "n'oublie pas la contribution de « nemo »": float(bm25_b45(query=["nautilus"])[0]),
          "relis la formule : le dénominateur dépend de la longueur relative du document": float(SCORES_B0_B45[0]),
          "relis la formule : l'idf de la docstring a un « 1 + »": float(SCORES_ORIGINAL_B45[0])})
wb.record("B4.5f", RANKING_B45, mistakes={"un document court peut-il battre un document qui contient le terme plus souvent ? Compare c) et d)": [0, 2, 1],
          "on classe du meilleur au moins bon": RANKING_B45[::-1]})
wb.record("B4.5g", False, mistakes={"recalcule les scores des docs 1 et 2 avec b = 0": True})
wb.record("B4.5h", float(SCORES_K0_B45[0]), decimals=4, mistakes={"relis la question : k₁ = 0, la fréquence ne compte plus": float(SCORES_B45[0]),
          "relis la question : on additionne les contributions des deux termes": float(SCORES_K0_B45[0] - IDF_NAUTILUS_B45)})
wb.record("B4.5i", True, mistakes={"calcule l'idf original de « nautilus », puis le score du doc 0": False})

WB_ANSWER {"decimals": 4, "hash": "b6522a8be0f0bb236d3f9c83370f04f558a18c6227065d6f42201a760341f600", "hash_coarse": "a48b7d23d5e59d27712d1dcc1e059c58989e04f695be6a154ede2eaa02ca9226", "id": "B4.5a", "kind": "float", "magnitude": 0, "mistakes": {"13710febca9f76f4944a1d322e4a239c0716cd6cd2f84e1f1623a96271bfae49": "relis la question : la longueur moyenne, pas totale", "c50e5d972154a1dce41aedad101c91d795261a72a74ed41431770dac72dab79d": "compte à nouveau les mots de chaque document"}, "sign": 1}


📝 Ex B4.5a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "c20c86da620b1753765885d8d50fdc620ff0a9812275cb8437868cc78a0bad43", "hash_coarse": "da5c14202078836e299d49330fa5809c7776ca74bc9c715e9f1901bfba132d92", "id": "B4.5b", "kind": "float", "magnitude": -1, "mistakes": {"4a676091a25a3ef092cb7c07a8814988b934c5b919d31f9f41f56ffda41c28b5": "relis la formule : l'idf de la docstring a un « 1 + »", "8c57528fd119936032f8e598247154bc865299dc925a40f25ff2a4e392a32a3e": "relis la formule : n_t + 0,5 au dénominateur", "e24d526e3fe0f87cf5ac697d432668b5eba141fc6efde153b17aba0ee68442d8": "le logarithme est naturel ici, pas décimal", "e8a5d9a283e790cdb715f590eff112818e135529f07c20c39ba452306c6ec0c5": "n_t compte les documents qui contiennent le terme, pas ses occurrences"}, "sign": 1}
📝 Ex B4.5b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "698342560e501f363f52fa195eec2235ccf23a74d98e52cf46bbe104c6927a10", "hash_coarse": "ac2cc1c82d0

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire si un modèle tient en mémoire, poids et KV cache compris, selon son dtype ?
2. Sais-tu appliquer à la main un chat template et compter les tokens d'un prompt ?
3. Sais-tu calculer BM25, recall@k et MRR à la main ?
4. Sais-tu compter les paramètres de LoRA et dériver les gradients de $\mathbf{A}$ et $\mathbf{B}$ ?

**Pour aller plus loin** : le cours LLM de Hugging Face et l'article de LoRA, cités dans la fiche. La suite : le bonus B5, les modèles génératifs d'images.

*Les exercices de code B4.12 à B4.30 (usage et prompting, recherche et RAG, fine-tuning, livraison) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B4` pour obtenir le notebook complet.*